In [8]:
import pandas as pd

In [9]:
filepath = "nrc_data_csv/russian_nrc.csv"
df = pd.read_csv(filepath)
df.head(12)

,word,anger,disgust,fear,joy,sadness,surprise
0,ошеломленный,0,0,0,0,0,0
1,счеты,0,0,0,0,0,0
2,покидать,0,0,1,0,1,0
3,заброшенный,1,0,1,0,1,0
4,оставление,1,0,1,0,1,1
5,уменьшаться,0,0,0,0,0,0
6,сокращение,0,0,0,0,0,0
7,абба,0,0,0,0,0,0
8,аббат,0,0,0,0,0,0
9,сокращать,0,0,0,0,0,0


In [10]:
from datasets import load_dataset
import pandas as pd

target_emotions = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
random_state = 42

# Load German evaluation data

comb_df = pd.concat(
    map(pd.read_csv, ["xed_russian.csv", "brighter_russian.csv"]),
    ignore_index=True
)

comb_df["text"] = comb_df["text"].astype(str)
comb_df["emotion"] = comb_df["emotion"].astype(str)

print("Original German rows:", len(comb_df))
print("\nOriginal German class distribution:")
print(comb_df["emotion"].value_counts())


# Balance German data
target_n = comb_df["emotion"].value_counts().min()

balanced_parts = []
for emotion, group in comb_df.groupby("emotion"):
    balanced_parts.append(
        group.sample(n=target_n, random_state=random_state)
    )

comb_df = (
    pd.concat(balanced_parts, ignore_index=True)
    .sample(frac=1, random_state=random_state)
    .reset_index(drop=True)
)

print("\nBalanced German rows:", len(comb_df))
print("\nBalanced German class distribution:")
print(comb_df["emotion"].value_counts())

Original German rows: 4540

Original German class distribution:
emotion
anger       1043
joy         1015
fear         734
sadness      724
surprise     612
disgust      412
Name: count, dtype: int64

Balanced German rows: 2472

Balanced German class distribution:
emotion
fear        412
surprise    412
sadness     412
joy         412
anger       412
disgust     412
Name: count, dtype: int64


In [11]:
import spacy
nlp = spacy.load("ru_core_news_sm")

print(df)

                word  anger  disgust  fear  joy  sadness  surprise
0       ошеломленный      0        0     0    0        0         0
1              счеты      0        0     0    0        0         0
2           покидать      0        0     1    0        1         0
3        заброшенный      1        0     1    0        1         0
4         оставление      1        0     1    0        1         1
...              ...    ...      ...   ...  ...      ...       ...
14149           зона      0        0     0    0        0         0
14150        зоопарк      0        0     0    0        0         0
14151  зоологический      0        0     0    0        0         0
14152       зоология      0        0     0    0        0         0
14153            зум      0        0     0    0        0         0

[14154 rows x 7 columns]


In [12]:
import random

random.seed(42)

df["word"] = df["word"].astype(str).str.lower().str.strip()

df = (
    df.groupby("word", as_index=False)[target_emotions]
    .max()
)

lexicon = df.set_index("word")[target_emotions].to_dict(orient="index")

emotion_frequency = {}

for word, scores in lexicon.items():
    active_count = sum(scores[emotion] for emotion in target_emotions)

    if active_count > 0:
        emotion_frequency[word] = 1 / active_count
    else:
        emotion_frequency[word] = 0

predictions = []

for text in comb_df["text"]:

    count = {emotion: 0 for emotion in target_emotions}

    doc = nlp(text)

    for token in doc:

        candidates = [
            token.lemma_.lower().strip() if token.lemma_ else "",
            token.text.lower().strip()
        ]

        for candidate in candidates:

            if candidate in lexicon:

                word_weight = emotion_frequency[candidate]

                for emotion in target_emotions:
                    count[emotion] += (
                        lexicon[candidate][emotion] * word_weight
                    )

                break

    maxim = max(count.values())

    print(text)
    print(count)
    print()

    if maxim == 0:
        pred = "unknown"
    else:
        best_emotions = [emo for emo, score in count.items() if score == maxim]
        pred = random.choice(best_emotions)

    predictions.append(pred)

comb_df["prediction"] = predictions


всё совсем страшно (
{'anger': 0.0, 'disgust': 0.0, 'fear': 0.5, 'joy': 0.0, 'sadness': 0.5, 'surprise': 0.0}

мне вк парень подарок прислал за 10 голосов. я в шоке. мы с ним общались 2 раза в 2012 году
{'anger': 0.3333333333333333, 'disgust': 0.0, 'fear': 0.3333333333333333, 'joy': 0.5, 'sadness': 0.0, 'surprise': 0.8333333333333333}

ну бляяяя всего 15 минут и огромная толпа(((
{'anger': 0.5, 'disgust': 0.0, 'fear': 0.5, 'joy': 0.0, 'sadness': 0.0, 'surprise': 0.0}

выходные пройдут на ураа!!! особенно с теми,с кем не виделась очень )давно,больше месяца
{'anger': 0.0, 'disgust': 0.5, 'fear': 0.0, 'joy': 0.0, 'sadness': 1.0, 'surprise': 0.5}

ну вас нахуй, я уебываю
{'anger': 0, 'disgust': 0, 'fear': 0, 'joy': 0, 'sadness': 0, 'surprise': 0}

А что если я не смогу снять его?
{'anger': 0, 'disgust': 0, 'fear': 0, 'joy': 0, 'sadness': 0, 'surprise': 0}

интересно гуф когда-нибудь исчезнет и перестанет читать, а то у него треки стали слишком отвратительны, полное дерьмо...
{'anger': 0.75

In [13]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np

target_emotions = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

def evaluate_lexicon_predictions(df, gold_col="emotion", pred_col="prediction"):
    """
    Evaluates lexicon predictions.
    Unknown predictions are counted as wrong in the main evaluation.
    Also reports coverage and covered-only performance.
    """

    eval_df = df.copy()
    eval_df[gold_col] = eval_df[gold_col].astype(str)
    eval_df[pred_col] = eval_df[pred_col].astype(str)

    # -------------------------
    # Main evaluation: unknown = wrong
    # -------------------------
    y_true = eval_df[gold_col]
    y_pred = eval_df[pred_col]

    print("=== Main evaluation: unknown counted as wrong ===")
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Balanced accuracy:", round(balanced_accuracy_score(y_true, y_pred), 4))
    print("Macro F1:", round(f1_score(y_true, y_pred, labels=target_emotions, average="macro"), 4))
    print("Weighted F1:", round(f1_score(y_true, y_pred, labels=target_emotions, average="weighted"), 4))
    print()

    print(classification_report(
        y_true,
        y_pred,
        labels=target_emotions,
        digits=3,
        zero_division=0
    ))

    cm = confusion_matrix(y_true, y_pred, labels=target_emotions)
    cm_df = pd.DataFrame(cm, index=target_emotions, columns=target_emotions)

    print("Confusion matrix:")
    print(cm_df)
    print()

    # -------------------------
    # Coverage
    # -------------------------
    covered_mask = eval_df[pred_col] != "unknown"
    coverage = covered_mask.mean()

    print("=== Coverage ===")
    print("Coverage:", round(coverage, 4))
    print("Covered examples:", covered_mask.sum())
    print("Total examples:", len(eval_df))
    print()

    # -------------------------
    # Covered-only evaluation
    # -------------------------
    covered_df = eval_df[covered_mask].copy()

    if len(covered_df) > 0:
        print("=== Covered-only evaluation ===")
        print("Accuracy:", round(accuracy_score(covered_df[gold_col], covered_df[pred_col]), 4))
        print("Balanced accuracy:", round(balanced_accuracy_score(covered_df[gold_col], covered_df[pred_col]), 4))
        print("Macro F1:", round(f1_score(
            covered_df[gold_col],
            covered_df[pred_col],
            labels=target_emotions,
            average="macro",
            zero_division=0
        ), 4))
        print()

        print(classification_report(
            covered_df[gold_col],
            covered_df[pred_col],
            labels=target_emotions,
            digits=3,
            zero_division=0
        ))
    else:
        print("No covered examples.")

    results = {
        "accuracy_unknown_wrong": accuracy_score(y_true, y_pred),
        "balanced_accuracy_unknown_wrong": balanced_accuracy_score(y_true, y_pred),
        "macro_f1_unknown_wrong": f1_score(y_true, y_pred, labels=target_emotions, average="macro", zero_division=0),
        "weighted_f1_unknown_wrong": f1_score(y_true, y_pred, labels=target_emotions, average="weighted", zero_division=0),
        "coverage": coverage,
    }

    if len(covered_df) > 0:
        results.update({
            "accuracy_covered_only": accuracy_score(covered_df[gold_col], covered_df[pred_col]),
            "balanced_accuracy_covered_only": balanced_accuracy_score(covered_df[gold_col], covered_df[pred_col]),
            "macro_f1_covered_only": f1_score(
                covered_df[gold_col],
                covered_df[pred_col],
                labels=target_emotions,
                average="macro",
                zero_division=0
            ),
        })

    return results

In [14]:
results = evaluate_lexicon_predictions(
    comb_df,
    gold_col="emotion",
    pred_col="prediction"
)

=== Main evaluation: unknown counted as wrong ===
Accuracy: 0.1998
Balanced accuracy: 0.1998
Macro F1: 0.2509
Weighted F1: 0.2509

              precision    recall  f1-score   support

       anger      0.231     0.068     0.105       412
     disgust      0.461     0.199     0.278       412
        fear      0.492     0.398     0.440       412
         joy      0.410     0.221     0.287       412
     sadness      0.276     0.204     0.235       412
    surprise      0.302     0.109     0.160       412

   micro avg      0.378     0.200     0.261      2472
   macro avg      0.362     0.200     0.251      2472
weighted avg      0.362     0.200     0.251      2472

Confusion matrix:
          anger  disgust  fear  joy  sadness  surprise
anger        28       48    34   23       39        10
disgust      36       82    60   29       51        29
fear         16       13   164   10       87        11
joy           8       10    16   91       13        39
sadness      17       17    32   

/Users/arsenimuravjov/miniforge3/envs/xed-llm/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/arsenimuravjov/miniforge3/envs/xed-llm/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
